# Identifying stage-specific miRNA signatures via clustering

### Clustering methods
- **K-Means**
- **Spectral Clustering**
- **Gaussian Mixture Model (GMM)**
- **Agglomerative (Hierarchical) Clustering**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

In [2]:
df = pd.read_csv("breast_cancer.csv")
label_col = "Label"
out_dir = Path("TeamsExports/Clustering")

stage_map = {1: "Stage_I", 2: "Stage_II", 3: "Stage_III", 4: "Stage_IV"}

for method_name in ["KM", "SC", "GMM", "HC"]:
    (out_dir / method_name).mkdir(parents=True, exist_ok=True)

In [3]:
def remove_constant_features(data_T):
    variances = data_T.var(axis=1)
    constant_mask = variances > 0
    return data_T[constant_mask]

def select_cluster(data_scaled, labels):
    unique, counts = np.unique(labels, return_counts=True)
    sizes = dict(zip(unique, counts))
    
    if len(sizes) == 1:
        return unique[0]
    
    c0_size, c1_size = sizes.get(0, 0), sizes.get(1, 0)
    
    if c0_size < 10:
        return 1
    if c1_size < 10:
        return 0
    
    return 0 if c0_size < c1_size else 1

def get_dynamic_n_neighbors(n_points):
    return max(10, int(np.sqrt(n_points)))

In [4]:
for stage_code, stage_slug in stage_map.items():
    print(f"\n{stage_slug}:")
    
    df_stage = df[df[label_col].isin([0, stage_code])].copy()
    data = df_stage.iloc[:, :-1]
    
    data_T = data.T.copy()
    data_T_filtered = remove_constant_features(data_T)
    
    n_points = data_T_filtered.shape[0]
    n_neighbors = get_dynamic_n_neighbors(n_points)
    
    scaler = StandardScaler()
    data_T_scaled = scaler.fit_transform(data_T_filtered.T).T
    
    methods = {
        "KM": lambda: KMeans(n_clusters=2, random_state=42, n_init=10),
        "SC": lambda: SpectralClustering(n_clusters=2, random_state=42,
                                         affinity="nearest_neighbors",
                                         n_neighbors=n_neighbors),
        "GMM": lambda: GaussianMixture(n_components=2, covariance_type="full",
                                        reg_covar=1e-6, random_state=42),
        "HC": lambda: AgglomerativeClustering(n_clusters=2, linkage="ward")
    }
    
    for method_name, method_func in methods.items():
        model = method_func()
        cluster_labels = model.fit_predict(data_T_scaled)
        selected_cluster = select_cluster(data_T_scaled, cluster_labels)
        selected_mirnas = data_T_filtered.index[cluster_labels == selected_cluster]
        
        print(f"  {method_name}: {len(selected_mirnas)} miRNAs selected")
        
        out_file = out_dir / method_name / f"{method_name}_selected_{stage_slug}.csv"
        pd.Series(selected_mirnas, name="miRNA").to_csv(out_file, index=False)


Stage_I:
  KM: 553 miRNAs selected
  SC: 61 miRNAs selected
  GMM: 627 miRNAs selected
  HC: 45 miRNAs selected

Stage_II:
  KM: 108 miRNAs selected
  SC: 48 miRNAs selected
  GMM: 580 miRNAs selected
  HC: 57 miRNAs selected

Stage_III:
  KM: 524 miRNAs selected
  SC: 110 miRNAs selected
  GMM: 578 miRNAs selected
  HC: 80 miRNAs selected

Stage_IV:
  KM: 100 miRNAs selected
  SC: 102 miRNAs selected
  GMM: 687 miRNAs selected
  HC: 70 miRNAs selected
